In [ ]:
import numpy as np

""""DP based Policy Iteration for a simple GridWorld environment"""
class SimpleGridWorld:
    """Simple 3x3 GridWorld for Policy Iteration demonstration"""
    
    def __init__(self):
        self.size = 3
        self.num_states = self.size * self.size
        self.num_actions = 4
        self.actions = ['up', 'down', 'left', 'right']
        
        # State mapping: (row, col) -> state_id
        self.state_to_id = {(i, j): i * self.size + j for i in range(self.size) for j in range(self.size)}
        self.id_to_state = {v: k for k, v in self.state_to_id.items()}
        
        # Terminal states
        self.terminal_states = [0, 8]  # Top-left and bottom-right corners
        
        # Initialize transition probabilities P[s][a][s'] and rewards R[s][a][s']
        self.P = np.zeros((self.num_states, self.num_actions, self.num_states))
        self.R = np.zeros((self.num_states, self.num_actions, self.num_states))
        
        self._build_transition_model()
    
    def _build_transition_model(self):
        """Build transition probabilities and rewards"""
        for s in range(self.num_states):
            if s in self.terminal_states:
                # Terminal states: stay in same state with 0 reward
                for a in range(self.num_actions):
                    self.P[s][a][s] = 1.0
                    self.R[s][a][s] = 0.0
            else:
                row, col = self.id_to_state[s]
                
                for a in range(self.num_actions):
                    # Determine next state based on action
                    if self.actions[a] == 'up':
                        next_row, next_col = max(0, row - 1), col
                    elif self.actions[a] == 'down':
                        next_row, next_col = min(self.size - 1, row + 1), col
                    elif self.actions[a] == 'left':
                        next_row, next_col = row, max(0, col - 1)
                    elif self.actions[a] == 'right':
                        next_row, next_col = row, min(self.size - 1, col + 1)
                    
                    next_state = self.state_to_id[(next_row, next_col)]
                    
                    # Set transition probability
                    self.P[s][a][next_state] = 1.0
                    
                    # Set reward
                    if next_state == 8:  # Reached goal (bottom-right)
                        self.R[s][a][next_state] = 10.0
                    else:
                        self.R[s][a][next_state] = -1.0  # Small penalty for each step



In [ ]:
class PolicyIteration:
    """Policy Iteration Algorithm using Dynamic Programming"""
    
    def __init__(self, env, gamma=0.9, theta=1e-6):
        self.env = env
        self.gamma = gamma  # Discount factor
        self.theta = theta  # Convergence threshold
        
        # Initialize random policy (each state has equal probability for each action)
        self.policy = np.random.randint(0, env.num_actions, env.num_states)
        
        # Initialize value function
        self.V = np.zeros(env.num_states)
    
    def policy_evaluation(self):
        """Policy Evaluation: Calculate value function for current policy"""
        print("Running Policy Evaluation...")
        
        iteration = 0
        while True:
            iteration += 1
            delta = 0
            old_V = self.V.copy()
            
            # Update value for each state
            for s in range(self.env.num_states):
                v = self.V[s]
                
                # Calculate new value using Bellman equation
                action = self.policy[s]
                new_value = 0
                
                for next_s in range(self.env.num_states):
                    prob = self.env.P[s][action][next_s]
                    reward = self.env.R[s][action][next_s]
                    new_value += prob * (reward + self.gamma * self.V[next_s])
                
                self.V[s] = new_value
                delta = max(delta, abs(v - self.V[s]))
            
            print(f"  Iteration {iteration}: max change = {delta:.6f}")
            
            # Check for convergence
            if delta < self.theta:
                break
        
        print(f"Policy evaluation converged after {iteration} iterations")
    
    def policy_improvement(self):
        """Policy Improvement: Update policy to be greedy w.r.t. current value function"""
        print("Running Policy Improvement...")
        
        policy_stable = True
        
        for s in range(self.env.num_states):
            old_action = self.policy[s]
            
            # Find best action for this state
            action_values = np.zeros(self.env.num_actions)
            
            for a in range(self.env.num_actions):
                for next_s in range(self.env.num_states):
                    prob = self.env.P[s][a][next_s]
                    reward = self.env.R[s][a][next_s]
                    action_values[a] += prob * (reward + self.gamma * self.V[next_s])
            
            # Choose action with highest value
            best_action = np.argmax(action_values)
            self.policy[s] = best_action
            
            # Check if policy changed
            if old_action != best_action:
                policy_stable = False
        
        return policy_stable
    
    def run_policy_iteration(self):
        """Main Policy Iteration Algorithm"""
        print("="*50)
        print("POLICY ITERATION ALGORITHM")
        print("="*50)
        
        iteration = 0
        
        while True:
            iteration += 1
            print(f"\n--- ITERATION {iteration} ---")
            
            # Step 1: Policy Evaluation
            self.policy_evaluation()
            
            # Step 2: Policy Improvement
            policy_stable = self.policy_improvement()
            
            # Print current results
            self.print_results()
            
            # Check for convergence
            if policy_stable:
                print(f"\nPolicy converged after {iteration} iterations!")
                break
        
        print("\n" + "="*50)
        print("FINAL OPTIMAL POLICY AND VALUES")
        print("="*50)
        self.print_results()
    
    def print_results(self):
        """Print current policy and value function"""
        action_symbols = ['↑', '↓', '←', '→']
        
        print("\nCurrent Policy:")
        for i in range(self.env.size):
            row = []
            for j in range(self.env.size):
                state_id = self.env.state_to_id[(i, j)]
                if state_id in self.env.terminal_states:
                    row.append("TERM")
                else:
                    action = self.policy[state_id]
                    row.append(f" {action_symbols[action]} ")
            print(" ".join(row))
        
        print("\nValue Function:")
        for i in range(self.env.size):
            row = []
            for j in range(self.env.size):
                state_id = self.env.state_to_id[(i, j)]
                row.append(f"{self.V[state_id]:6.2f}")
            print(" ".join(row))



In [7]:

def main():
    """Policy Iteration Hello World Example"""
    
    print("POLICY ITERATION - HELLO WORLD EXAMPLE")
    print("="*50)
    print("Environment: 3x3 GridWorld")
    print("Start: Top-left (0,0)")
    print("Goal: Bottom-right (2,2)")
    print("Actions: up, down, left, right")
    print("Rewards: +10 for reaching goal, -1 for each step")
    print("Terminal states: (0,0) and (2,2)")
    
    # Create environment
    env = SimpleGridWorld()
    
    # Create and run Policy Iteration agent
    pi_agent = PolicyIteration(env, gamma=0.9)
    pi_agent.run_policy_iteration()
    
    print("\nPolicy Iteration completed successfully!")
    print("The agent learned the optimal policy to reach the goal!")


if __name__ == "__main__":
    main()

POLICY ITERATION - HELLO WORLD EXAMPLE
Environment: 3x3 GridWorld
Start: Top-left (0,0)
Goal: Bottom-right (2,2)
Actions: up, down, left, right
Rewards: +10 for reaching goal, -1 for each step
Terminal states: (0,0) and (2,2)
POLICY ITERATION ALGORITHM

--- ITERATION 1 ---
Running Policy Evaluation...
  Iteration 1: max change = 1.900000
  Iteration 2: max change = 0.900000
  Iteration 3: max change = 0.810000
  Iteration 4: max change = 0.729000
  Iteration 5: max change = 0.656100
  Iteration 6: max change = 0.590490
  Iteration 7: max change = 0.531441
  Iteration 8: max change = 0.478297
  Iteration 9: max change = 0.430467
  Iteration 10: max change = 0.387420
  Iteration 11: max change = 0.348678
  Iteration 12: max change = 0.313811
  Iteration 13: max change = 0.282430
  Iteration 14: max change = 0.254187
  Iteration 15: max change = 0.228768
  Iteration 16: max change = 0.205891
  Iteration 17: max change = 0.185302
  Iteration 18: max change = 0.166772
  Iteration 19: max ch